In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
from tqdm import tqdm

CRS = "EPSG:31370"  # Belgian Lambert 72
tqdm.pandas()
rng = np.random.default_rng(42)

In [ ]:
# Load all addresses with their sectors
addresses_fp = "input_data/urbis_addresses.shp"
addresses = gpd.read_file(addresses_fp)

# Rename sector field for clarity
addresses_with_sector = addresses.rename(columns={"STATNISCOD": "sector_id"})

# Load synthetic population
pop = pd.read_csv("../1_synthetic_population/output/employed_population.csv")

In [ ]:
# Get a dictionary of municipality_id to sector_id, used for ZZZZ sectors
sector_ids = pd.Series(addresses_with_sector["sector_id"].dropna().astype(str).unique())
municipality_to_sector = sector_ids.groupby(sector_ids.str[:5]).apply(list).to_dict()

In [ ]:
# Expand population to individual agents
pop = pop.loc[pop.index.repeat(pop["population"])].copy()
pop.drop(columns=["population"], inplace=True)
pop.reset_index(drop=True, inplace=True)
print(f"Expanded to individual agents: {len(pop)} rows")

In [ ]:
# Load land use data
land_use = gpd.read_file("input_data/pras.shp")

# Filter for residential land use types
home_land_use = land_use[
    ~land_use["AFFECTATIO"].isin(
        [
            'AfEau', 'AfZAgric', 'AfZBio', 'AfZChFer', 'AfZCimetiere',
            'AfZForest', 'AfZIndustrie', 'AfZParc', 'AfZParcKR', 'AfZSport', 'AfZTransPort', 'AfZVerte'
        ]
    )
]

# Ensure both use the same CRS format (EPSG:31370 = Belgian Lambert 72)
addresses = addresses.to_crs(CRS)
home_land_use = home_land_use.to_crs(CRS)

# Merge addresses with land use data and filter for home zones
addresses_land_use = addresses.sjoin(home_land_use, how="inner")

addresses_with_sector = addresses_land_use.rename(columns={"STATNISCOD": "sector_id"})

In [ ]:
# Check sectors without addresses
unique_sectors_in_addresses = set(addresses_with_sector["sector_id"].unique())
unique_sectors_in_population = set(pop["sector"].unique())

print([sector for sector in unique_sectors_in_population if sector not in unique_sectors_in_addresses and not sector.endswith("ZZZZ")])

In [ ]:
# Check how many people live in sectors without addresses
pop["has_no_address"] = ~pop["sector"].isin(unique_sectors_in_addresses) & ~pop["sector"].str.endswith("ZZZZ", na=False)
num_without_address = len(pop[pop["has_no_address"]])
print(f"Number of agents without address (excluding ZZZZ): {num_without_address} ({num_without_address / len(pop) * 100:.2f}%)")

In [ ]:
# Create cyclic assignment of addresses to agents within each sector
# First, create a dictionary of addresses per sector
# Group by sector AND coordinate to get unique physical locations
# There are some addresses that only vary by boxnumber, eg 21001A41-, 144113,3699999, 168537,63899999, boxnumber 0080
sector_addresses = {}
for sector_id in addresses_with_sector["sector_id"].unique():
    sector_data = addresses_with_sector[addresses_with_sector["sector_id"] == sector_id]
    # Group by coordinates to get unique physical locations, keep first address per location
    unique_locations = sector_data.drop_duplicates(
        subset=["geometry"]
    ).geometry.tolist()
    sector_addresses[sector_id] = unique_locations

# Load all Brussels statistical sectors
statistical_sectors = gpd.read_file(
    "input_data/urbis_statistical_sectors.shp"
)
statistical_sectors = statistical_sectors.to_crs(CRS)

# For sectors without home addresses assign centroid of the sector
for sector in statistical_sectors["NISCODE"].unique():
    if sector not in sector_addresses:
        sector_geom = statistical_sectors[
                statistical_sectors["NISCODE"] == sector
            ]
        if not sector_geom.empty:
            centroid = sector_geom.geometry.centroid.values[0]
            sector_addresses[sector] = [centroid]


# Shuffle addresses per sector and create cyclic assignment
sector_addresses_shuffled = {
    sector_id: rng.permutation(addresses).tolist()
    for sector_id, addresses in sector_addresses.items()
}

# Create a counter for each sector to track cyclic position
sector_counters = {sector_id: 0 for sector_id in sector_addresses_shuffled.keys()}

# Get a list of all available sectors in case some sector is ZZZZ
all_available_sectors = list(sector_addresses_shuffled.keys())

In [ ]:
# Assign statistical sector (if needed)
def assign_sector(sector_id):
    """
    Keep original sector unless it is a ends with ZZZZ.
    For ZZZZ sectors, sample a valid sector from the same municipality.
    """
    if pd.isna(sector_id):
        return rng.choice(all_available_sectors)

    if sector_id.endswith("ZZZZ"):
        municipality_id = sector_id[:5]
        possible_sectors = municipality_to_sector.get(municipality_id, [])

        if len(possible_sectors) > 0:
            return rng.choice(possible_sectors)

        # Fallback if municipality lookup is missing
        return rng.choice(all_available_sectors)

    return sector_id

# Check the number fo agents with ZZZZ sectors before assignment
num_zzzz_before = pop["sector"].str.endswith("ZZZZ").sum()
print(f"Number of agents with ZZZZ sectors before assignment: {num_zzzz_before}")

# Keep original sector in `sector`, and use `assigned_sector` for address sampling
pop["assigned_sector"] = pop["sector"].apply(assign_sector)

In [ ]:
# Assign address
def assign_cyclic_address(sector_id):
    if sector_id in sector_addresses_shuffled:
        addresses_list = sector_addresses_shuffled[sector_id]
        idx = sector_counters[sector_id] % len(addresses_list)
        sector_counters[sector_id] += 1
        return addresses_list[idx]
    else:
        print(f"No addresses found for sector {sector_id}")
        return None


# Shuffle the population by sector before assignment
pop = pop.sample(frac=1).reset_index(drop=True)

print("Assigning random addresses to agents...")
pop["home_geometry"] = pop["assigned_sector"].progress_apply(assign_cyclic_address)

In [ ]:
# Drop the three agents with no address
pop = pop[pop['assigned_sector'] != '21001A071'].copy()
pop = pop[pop['assigned_sector'] != '21010A29-'].copy()

In [ ]:
# Convert to GeoDataFrame with coordinates
pop_gdf = gpd.GeoDataFrame(pop, geometry="home_geometry", crs=addresses.crs)

# Extract X/Y coordinates if needed
pop_gdf["home_x"] = pop_gdf.geometry.x
pop_gdf["home_y"] = pop_gdf.geometry.y

# Save the result
pop_gdf.to_file("output/employed_population_with_home_address.shp")
pop_gdf[
    [
        "sex",
        "age",
        "education",
        "industry",
        "professional_status", 
        "sector",
        "assigned_sector",
        "home_x",
        "home_y",
    ]
].to_csv("output/employed_population_with_home_address.csv", index=False)